# Lesson 1: One call to the model

**Project:** Learn-AI-Agent, building toward a work log agent
**Model:** `qwen3:8b`, running on this PC through Ollama

## What we are looking for

By the end of this notebook you should be able to say yes to four things:

1. Python can reach the model running on this PC.
2. I know what goes in (a list of messages) and what comes back (a response object).
3. I know the model remembers nothing between calls unless I send the history again.
4. I have one small function, `ask_model`, that every later lesson will reuse.

Nothing here is an agent yet. This is the single building block an agent repeats in a loop.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter** to run it and move to the next one. Run the cells in order from top to bottom. If PyCharm asks for an interpreter or kernel, choose this project's `.venv`.

## Step 1: Check that Python can see Ollama

Ollama runs as a small server in the background on this PC. The `ollama` Python package talks to it. This cell asks the server which models are downloaded.

In [1]:
import ollama

MODEL = "qwen3:8b"

downloaded = ollama.list()
for m in downloaded.models:
    size_gb = m.size / 1e9
    print(f"{m.model:<20} {size_gb:.1f} GB")

qwen3:8b             5.2 GB


### How to read it

One line per downloaded model, with its size on disk. You should see `qwen3:8b` at about 5.2 GB.

### What it means

Python reached the Ollama server and the model is available. If this cell fails with a connection error, Ollama is not running: start it from the Windows Start menu and run the cell again.

`MODEL` is set once here so every later cell uses the same model name.

## Step 2: The first call

This sends one message to the model and prints the reply.

The first run can take 10 to 30 seconds, because the model has to load onto the video cards. Later calls are faster.

In [2]:
messages = [
    {"role": "user", "content": "In one sentence, what is a work log?"}
]

response = ollama.chat(model=MODEL, messages=messages, think=False)

print(response.message.content)

A work log is a record of an individual's activities, tasks, and time spent on various projects or responsibilities during a specific period.


### How to read it

One sentence from the model describing a work log. The exact wording will differ each time you run it.

### What it means

Three pieces matter here:

- **`messages`** is a list. Each item is a dictionary with a `role` (who is speaking) and `content` (what they said). Here there is one message from the `user`.
- **`ollama.chat(...)`** sends that list to the model and waits for the reply.
- **`response.message.content`** is the text of the reply.

`think=False` turns off this model's "thinking out loud" step, which keeps replies short and fast for these lessons.

## Step 3: Look at everything that came back

The reply text is only part of the response. The rest tells you how much work the model did and how fast.

In [3]:
print("role:         ", response.message.role)
print("reply:        ", response.message.content)
print()
print("tokens in:    ", response.prompt_eval_count)
print("tokens out:   ", response.eval_count)

seconds_generating = response.eval_duration / 1e9
print("speed:        ", round(response.eval_count / seconds_generating, 1), "tokens per second")
print("total time:   ", round(response.total_duration / 1e9, 1), "seconds")

role:          assistant
reply:         A work log is a record of an individual's activities, tasks, and time spent on various projects or responsibilities during a specific period.

tokens in:     26
tokens out:    28
speed:         21.3 tokens per second
total time:    22.6 seconds


### How to read it

- **role** is `assistant`. That is the model's own role name, the counterpart to `user`.
- **tokens in** is the size of what you sent. **tokens out** is the size of the reply.
- **speed** is how fast the reply was written.
- **total time** includes loading the model if it was not already loaded.

### What it means

A **token** is the unit a model reads and writes: a short word or a piece of a longer word. A rough rule is 4 tokens for every 3 English words.

Tokens matter for agents for two reasons. An agent calls the model many times, so slow tokens add up. And every model has a limit on how many tokens it can read at once, which caps how much history you can send.

For speed, roughly: 15 or more tokens per second means the model is on the video cards. Under 5 means it is probably running on the processor. The next step checks that directly.

## Step 4: Check where the model is running

This asks Ollama how much of the loaded model is sitting in video card memory.

In [4]:
running = ollama.ps()

if not running.models:
    print("No model is loaded right now. Run the Step 2 cell again, then this one.")

for m in running.models:
    on_gpu = m.size_vram / m.size * 100
    print(f"model:        {m.model}")
    print(f"total size:   {m.size / 1e9:.1f} GB")
    print(f"on the GPUs:  {m.size_vram / 1e9:.1f} GB  ({on_gpu:.0f}%)")

model:        qwen3:8b
total size:   5.9 GB
on the GPUs:  5.9 GB  (100%)


### How to read it

- **total size** is the memory the loaded model needs. It is larger than the 5.2 GB download because it includes working space.
- **on the GPUs** is how much of that is in video card memory. The percentage is the number to watch.

### What it means

- **100%** means the whole model is on the two GTX 1060 cards. This is the goal.
- **Less than 100%** means part of the model spilled into regular memory and runs on the processor, which is slower. The usual cause is other programs using video memory. Closing browsers or a virtual machine frees space.

Ollama unloads a model after about 5 idle minutes, so an empty result only means it has been a while since the last call.

## Step 5: The model has no memory

This is the most important idea in the lesson. Two calls: the first tells the model a fact, the second asks about it.

In [5]:
first = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "I spent this morning on my Cornerpost Watch project."}],
    think=False,
)
print("Call 1 reply:", first.message.content)
print()

second = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Which project did I work on this morning?"}],
    think=False,
)
print("Call 2 reply:", second.message.content)

Call 1 reply: That sounds like a great project! Could you tell me more about what you're working on with the Cornerpost Watch? Are you building it for a specific purpose, like security, timekeeping, or something else? I’d love to hear about your progress and maybe even help brainstorm or troubleshoot if you need it!

Call 2 reply: I don't have access to information about your personal work history or projects. However, if you'd like to share more details about your work or what you were doing this morning, I'd be happy to help you organize or reflect on it!


### How to read it

Call 1 gets a normal acknowledgment. Call 2 should say it does not know, or ask you which project. It may also guess, and any guess is made up.

### What it means

Each call starts from nothing. The model does not store your earlier message anywhere. The second call only saw one sentence, the question, so it had no way to answer.

A chat app feels like it remembers because the app sends the whole conversation again on every turn. The next step does the same thing by hand.

## Step 6: Memory is the message list

Same question, but this time the earlier exchange is sent along with it.

In [6]:
history = [
    {"role": "user", "content": "I spent this morning on my Cornerpost Watch project."},
    {"role": "assistant", "content": first.message.content},
    {"role": "user", "content": "Which project did I work on this morning?"},
]

third = ollama.chat(model=MODEL, messages=history, think=False)

print("Reply with history:", third.message.content)
print()
print("Messages sent:", len(history))
print("Tokens in:    ", third.prompt_eval_count)

Reply with history: You mentioned that you spent the morning on your **Cornerpost Watch project**. That was the project you were working on! If you'd like to share more details about it — like what it is, how it works, or what you're trying to achieve with it — I'd be happy to help you with ideas, troubleshooting, or even design suggestions!

Messages sent: 3
Tokens in:     110


### How to read it

The reply should now name Cornerpost Watch. **Messages sent** is 3, and **tokens in** is higher than in Step 3 because the model had to read the whole history.

### What it means

The model's "memory" is whatever is in the `messages` list for that one call. Your code decides what goes in it.

This is the heart of how an agent works. In later lessons the list will also hold the results of tools the agent ran, and the model will read those results to decide its next step. A longer list means more tokens to read on every call, which is why agents have to be careful about what they keep.

## Step 7: Give the model a job with a system message

A `system` message goes first in the list. It sets the model's role and rules for the whole conversation.

In [7]:
SYSTEM_PROMPT = (
    "You are a work log assistant. "
    "When the user describes work they did, reply with exactly one line in this form: "
    "LOGGED: <project> | <what was done>. "
    "If the message is not about work, reply with: NOT A LOG ENTRY."
)

tests = [
    "Fixed the alert email bug in Cornerpost Watch.",
    "Spent an hour cleaning the meat price data.",
    "What is the capital of Colorado?",
]

for text in tests:
    reply = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        think=False,
        options={"temperature": 0},
    )
    print("You said:", text)
    print("Model:   ", reply.message.content)
    print()

You said: Fixed the alert email bug in Cornerpost Watch.
Model:    LOGGED: Cornerpost Watch | Fixed the alert email bug

You said: Spent an hour cleaning the meat price data.
Model:    LOGGED: Meat Price Data | Cleaned the meat price data.

You said: What is the capital of Colorado?
Model:    NOT A LOG ENTRY



### How to read it

The first two replies should start with `LOGGED:` followed by a project and a short description. The third should be `NOT A LOG ENTRY`. A small model will sometimes drift from the exact format. Note any that do.

### What it means

There are now three roles: `system` (the standing instructions), `user` (you), and `assistant` (the model).

The system message is how you turn a general model into a specific helper. Here it produced something close to a log entry, but look at what did not happen: nothing was saved. The model only wrote text that looks like a record. To save an entry, the model needs a way to call your Python code. That is a **tool**, and it is the subject of lesson 2.

`temperature` set to 0 makes the model pick its most likely wording each time, so results are more repeatable. That is useful when testing.

## Step 8: Wrap the call in one function

Every later lesson calls the model the same way, so the call goes in a function. This is the `ask_model` from the loop sketch earlier in our chat.

In [8]:
def ask_model(messages, model=MODEL):
    """Send a list of messages to the local model and return its reply text."""
    response = ollama.chat(
        model=model,
        messages=messages,
        think=False,
        options={"temperature": 0},
    )
    return response.message.content


conversation = [{"role": "system", "content": "You are a helpful assistant. Answer in one short sentence."}]

for question in ["What is a token?", "Give me one example."]:
    conversation.append({"role": "user", "content": question})
    answer = ask_model(conversation)
    conversation.append({"role": "assistant", "content": answer})
    print("You:  ", question)
    print("Model:", answer)
    print()

print("Messages in the conversation now:", len(conversation))

You:   What is a token?
Model: A token is a digital representation of value used in blockchain and cryptocurrency systems to represent assets, rights, or utilities.

You:   Give me one example.
Model: Bitcoin is an example of a token, specifically a cryptocurrency token used as a medium of exchange.

Messages in the conversation now: 5


### How to read it

Two questions and two answers. The second answer should be an example of a token, which shows the model understood "one example" refers to the first question. The conversation ends with 5 messages: 1 system, 2 user, 2 assistant.

### What it means

The small `for` loop is a complete chat program: add the user's message, call the model, add the reply, repeat. The list grows by two messages each turn, and that growing list is the conversation's memory.

An agent's loop has the same shape. The difference is that some turns will be the model asking for a tool and your code adding the tool's result, with no person involved.

## Try it yourself

Change one thing at a time and rerun the cell:

1. In Step 7, add a fourth test sentence about one of your own projects. Does it get logged correctly?
2. In Step 7, change the system prompt so the format includes a time estimate, such as `LOGGED: <project> | <what was done> | <hours>`. What does the model do when you do not mention hours?
3. In Step 2, change `think=False` to `think=True`, then add a line `print(response.message.thinking)`. Compare the speed in Step 3 with and without thinking.

Exercise 2 shows a real weakness: when information is missing, a model may invent it. Keep that in mind for lesson 2.

## What it all means

- A model call is: a list of messages in, one message out.
- The model keeps nothing between calls. The message list is the only memory, and your code controls it.
- A system message gives the model a job and a format.
- The model can only produce text. It cannot save, look up, or change anything on its own.

## What is next and why

**Lesson 2: One tool.** You will write a Python function, `add_entry`, that saves a work log entry to a file, and describe it to the model. The model will then be able to ask for that function to be run with the project and description filled in.

That is the step from "text that looks like a record" to "a record that exists", and it is the first half of what makes an agent. Lesson 3 adds the loop, which is the second half.